# 07 · Heads-up no-limit: push/fold

This notebook solves the smallest no-limit game and checks the two claims the lesson makes: ranges widen as stacks shrink, and the caller's boundary follows the pot odds. It uses the repository's chance-sampled CFR solver with its default equity model, so these are teaching numbers, not published chart values.

In [1]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / 'pokerbot').is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError('Run this notebook inside the pokerbots repository')

A ten-big-blind game is `stack=20` in this model: each player posts a one-ante blind, so one big blind is two antes. The solver returns a shove probability for all 169 starting-hand classes and a call probability for the same 169 classes.

In [2]:
from pokerbot.solvers.push_fold import class_index, solve_push_fold

result = solve_push_fold(stack=20, iterations=200_000, seed=7)  # 10 bb

print(f"{len(result.shove_range)}/169 classes shove at 10 bb")
print(f"{len(result.call_range)}/169 classes call at 10 bb")
for hand in ("AA", "AKs", "72o"):
    i = class_index(hand)
    print(f"{hand}: shove {result.shove_threshold[i]:.1%}, call {result.call_threshold[i]:.1%}")
print(f"exact exploitability: {result.exploitability:.4f} antes")

108/169 classes shove at 10 bb
77/169 classes call at 10 bb
AA: shove 99.9%, call 99.9%
AKs: shove 99.9%, call 99.9%
72o: shove 0.0%, call 0.0%


exact exploitability: 0.0034 antes


Aces always shove and seven-deuce offsuit never does. The interesting part is the boundary, where the strategy mixes. Sorting the mixed hands by shove probability shows that the range edge is not a single clean threshold.

In [3]:
from pokerbot.solvers.push_fold import class_name

mixed = sorted(
    ((class_name(i), p) for i, p in result.shove_threshold.items() if 0.1 < p < 0.9),
    key=lambda item: -item[1],
)
print(f"{len(mixed)} mixed hands at 10 bb:")
for name, p in mixed:
    print(f"  {name}: shove {p:.1%}")

5 mixed hands at 10 bb:
  95s: shove 87.5%
  T6o: shove 86.5%
  94s: shove 76.4%
  98o: shove 60.0%
  T3s: shove 48.3%


## Stack depth moves both ranges

Solving again at several depths gives the classic shape: the small blind shoves fewer hands as the risk grows, and the big blind calls fewer hands as the shover's range tightens.

In [4]:
depths = (5, 10, 15, 20, 30, 40)
solved = {bb: solve_push_fold(stack=bb * 2, iterations=200_000, seed=7) for bb in depths}

print("stack   SB shoves   BB calls   exploitability")
for bb in depths:
    row = solved[bb]
    print(f"{bb:>2} bb   {len(row.shove_range):>3}/169      {len(row.call_range):>3}/169   {row.exploitability:.4f}")

stack   SB shoves   BB calls   exploitability


 5 bb   126/169      104/169   0.0019


10 bb   108/169       77/169   0.0034


15 bb    91/169       66/169   0.0051


20 bb    87/169       54/169   0.0060


30 bb    76/169       45/169   0.0088


40 bb    66/169       42/169   0.0129


The default equity function is a strength heuristic, not a measured preflop table. Real equities compress the extremes, so a solver fed a real table produces tighter ranges. The trend is the lesson; the exact boundary hands are approximate.

## What the game is worth

The small blind's value is the expected payoff of the solved profile, averaged over the combo-weighted class distribution. Because the model draws both classes independently, this is a direct sum rather than a sampled estimate.

In [5]:
from pokerbot.solvers.push_fold import N_CLASSES, combo_count, default_equity


def profile_value(result, stack, ante=1):
    total = sum(combo_count(i) for i in range(N_CLASSES))
    value = 0.0
    for i in range(N_CLASSES):
        shove = result.shove_threshold[i]
        fold = 1.0 - shove
        row = 0.0
        for j in range(N_CLASSES):
            call = result.call_threshold[j]
            showdown = stack * (2.0 * default_equity(i, j) - 1.0)
            row += combo_count(j) * (shove * ((1.0 - call) * ante + call * showdown)
                                     + fold * -ante)
        value += combo_count(i) * row
    return value / (total * total)


for bb in (5, 10, 20):
    print(f"{bb:>2} bb: small-blind value {profile_value(solved[bb], bb * 2) / 2:+.3f} bb")

 5 bb: small-blind value -0.063 bb
10 bb: small-blind value -0.128 bb
20 bb: small-blind value -0.200 bb


The value is negative at every depth, and it falls as the stacks get deeper. Both players post the same ante, so this is not a blind tax: it is the cost of acting first, which lets the big blind fold its worst hands after seeing the shove.

## The caller's break-even point

The big blind's decision is a pot-odds problem. Calling `bb - 0.5` big blinds to win a final pot of `2 * bb` requires `(bb - 0.5) / (2 * bb)` equity, which rises with depth. That is the analytic reason the calling range tightens.

In [6]:
from pokerbot.games import (NoLimitBettingRound, pot_fraction_for_geometric_growth,
                            pot_odds_call_fraction)

for bb in (5, 10, 20):
    call = bb - 0.5
    pot_before = bb + 0.5
    needed = pot_odds_call_fraction(pot_before, call)
    print(f"{bb:>2} bb: call {call:.1f} into {pot_before:.1f} -> {needed:.1%} equity needed")

print()
print("equal bet fraction, pot 10 to 100 over 3 streets:",
      round(pot_fraction_for_geometric_growth(10, 3, 100), 3))
round_ = NoLimitBettingRound([100, 100], pot=10, bet_menu=(0.33, 0.75, 1.5))
print("legal actions:", [str(action) for action in round_.legal_actions()])

 5 bb: call 4.5 into 5.5 -> 45.0% equity needed
10 bb: call 9.5 into 10.5 -> 47.5% equity needed
20 bb: call 19.5 into 20.5 -> 48.8% equity needed

equal bet fraction, pot 10 to 100 over 3 streets: 0.577
legal actions: ['check', 'wager to 1', 'wager to 3', 'wager to 8', 'wager to 15', 'wager to 100']


The betting-round engine is a separate building block: it turns pot fractions into legal wager sizes, enforces minimum raises, and returns unmatched chips when a bet is not called. A full no-limit hand still needs four of these rounds, a bet menu per node, and a solver that can re-solve subtrees.

## What this notebook does not do

- It does not use a measured preflop equity table, so the ranges are wider than published charts. Load one with `load_equity_table` to change that.
- It does not compute exploitability; a full best response over the 169 × 169 deal tree is left as an exercise.
- It does not re-solve, translate off-menu bets, or model postflop play. Those are the remaining Module 07 deliverables, and the [interactive lab](../docs/demos/pushfold-module-07.html) covers the charts interactively.